# Real-Time Data Ingestion with Spark Structured Streaming and PostgreSQL
**By:** Molo Munyansanga

This notebook runs the streaming pipeline step by step, so each stage can be seen:

1. **Part 1, the data:** look at the CSV files written by the data generator.
2. **Part 2, the transformations:** clean and validate a few files, one step at a time.
3. **Part 3, the stream:** start Spark Structured Streaming, watch it write to PostgreSQL in real time, and check the results.

The logic lives in the `src/` folder and is shared with `spark_streaming_to_postgres.py`, which runs the same pipeline from the terminal.

**You'll need two things running:**
- this notebook, for Spark;
- a second terminal for the data generator, `python data_generator.py`, started when the notebook asks for it.

> **Don't run `spark_streaming_to_postgres.py` while this notebook's stream is running.** Both would use the same checkpoint (Spark's record of which files it has processed).

## Setup

In [ ]:
# Let the notebook find the src/ folder (one level above this notebook)
import sys
sys.path.append("..")

import pandas as pd
from pyspark.sql import functions as F

from src import monitor
from src.config import BATCH_LOG, GENERATOR_LOG, INCOMING_DIR
from src.postgres_sink import read_query, table_counts
from src.schemas import RAW_EVENT_SCHEMA
from src.spark_session import get_spark
from src.streaming import reset_pipeline, start_stream
from src.transform import add_reject_reason, clean_events, normalize, parse_types, rejected_events, valid_events

spark = get_spark()
# Show Spark DataFrames as tables in the notebook
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 40)
print(f"Spark {spark.version}")

### Start from a clean slate
`reset_pipeline()` empties the three tables and deletes the incoming files, Spark's checkpoint and both logs, so the results below come from this run only.

In [ ]:
reset_pipeline()
table_counts()

---
## Part 1: The data

**Now open a second terminal** (with the virtual environment activated) and write three files:

```
python data_generator.py --files 3
```

The next cell waits until the files arrive.

In [ ]:
files = monitor.wait_for_files(min_files=3)

Spark reads the files with an explicit schema. Every column is read as **text** on purpose: if Spark read `price` as a number, a value like `"abc"` would silently become null and the original value would be lost. Reading text first lets the cleaning step decide what is valid.

In [ ]:
RAW_EVENT_SCHEMA.simpleString()

In [ ]:
raw = (
    spark.read.schema(RAW_EVENT_SCHEMA).option("header", True).csv(str(INCOMING_DIR))
    .withColumn("source_file", F.col("_metadata.file_name"))
)
print(f"{raw.count()} rows in {len(files)} files")
raw.limit(5)

The generator deliberately includes three kinds of imperfect rows. Its log records how many of each it wrote, which is the answer key for checking the pipeline later:

In [ ]:
pd.read_csv(GENERATOR_LOG)

---
## Part 2: The transformations

The cleaning functions in `src/transform.py` work the same on a normal DataFrame and a streaming one, so they can be tried here on the three files first.

### Step 1: Normalize the formatting
Remove extra spaces, lowercase `event_type` and `event_id`, and turn empty text into null. Each cleaned value goes into a new `clean_` column, so the original stays available. Below are the **messy** rows, before and after:

In [ ]:
normalized = normalize(raw)
messy = normalized.filter(
    (F.col("event_type") != F.col("clean_event_type")) | (F.col("product_name") != F.col("clean_product_name"))
)
messy.select("event_type", "clean_event_type", "product_name", "clean_product_name").limit(8)

### Step 2: Convert to the right types
`try_cast` and `try_to_timestamp` return null for values that can't be converted (e.g. a quantity of `"two"`), instead of stopping the stream with an error. Spark 4 runs in ANSI mode, where a plain `cast` would fail the whole batch.

In [ ]:
parsed = parse_types(normalized)
parsed.select("event_time", "parsed_event_time", "price", "parsed_price", "quantity", "parsed_quantity").limit(5)

### Step 3: Validate
Every row is checked against the rules below, and the **first** rule it breaks is recorded in `reject_reason` (null if it passes all of them). The rules match the constraints of the `events` table in PostgreSQL, so the database never receives a row it would refuse.

| Rule | Reason if broken |
|---|---|
| `event_id` present and a valid UUID | `missing_event_id` / `invalid_event_id` |
| `event_time` present and a valid timestamp | `missing_event_time` / `invalid_event_time` |
| `user_id` present and a whole number | `missing_user_id` / `invalid_user_id` |
| `session_id` present | `missing_session_id` |
| `event_type` is view, add_to_cart or purchase | `unknown_event_type` |
| `product_id` a whole number; `product_name`, `category` present | `invalid_product_id` / `missing_...` |
| `price` a number above 0 | `invalid_price` / `non_positive_price` |
| `quantity` a whole number above 0 | `invalid_quantity` / `non_positive_quantity` |

In [ ]:
checked = add_reject_reason(parsed)
checked.groupBy("reject_reason").count().orderBy(F.desc("count"))

The rejected rows, with their original values:

In [ ]:
rejected_events(checked).select("reject_reason", "event_time", "user_id", "event_type", "price", "quantity")

### Step 4: Shape the valid rows for the `events` table
Valid rows get their final column names and types, plus `total_amount` (price × quantity). This is `clean_events()` followed by `valid_events()`, the same functions the stream uses.

In [ ]:
valid = valid_events(clean_events(raw))
valid.printSchema()
valid.limit(5)

### Duplicates
Some valid rows are exact copies of earlier events. The stream removes duplicates within a batch with `dropDuplicates(["event_id"])`, and PostgreSQL skips any event already stored by an earlier batch (`ON CONFLICT DO NOTHING`), because `event_id` is the table's primary key.

In [ ]:
duplicates = valid.groupBy("event_id").count().filter("count > 1")
print(f"{valid.count()} valid rows, {valid.select('event_id').distinct().count()} unique events, "
      f"{duplicates.count()} event(s) appearing more than once")

---
## Part 3: The stream

### Start the stream
`start_stream()` builds the streaming query and starts it **in the background**, so the notebook stays usable:

- **Source:** the `data/incoming/` folder. Every 5 seconds (the *trigger*), Spark takes the files that arrived since the last check as one **micro-batch** (at most 20 files at a time).
- **Processing and sink:** `foreachBatch` passes each micro-batch to `write_batch()` in `src/postgres_sink.py`, which cleans it with the functions above and writes it to PostgreSQL. Valid rows go through the staging table into `events`, and invalid rows go to `rejected_events`.
- **Checkpoint:** Spark records which files it has processed in `data/checkpoints/`, so a restarted stream continues where it stopped instead of processing everything again.

The three files from Part 1 are already in the folder, so they become the first micro-batch.

In [ ]:
query = start_stream(spark)
query.status

### Watch it in real time
**Now start the generator in your second terminal, without a file limit:**

```
python data_generator.py
```

The next cell prints the stream's progress every 5 seconds for 40 seconds. You can run it again to keep watching.

In [ ]:
monitor.watch(query, seconds=40, every=5)

### Inside the latest micro-batch
`query.lastProgress` is Spark's report on the most recent batch: how many rows it read (`numInputRows`), how fast (`processedRowsPerSecond`), and how long each part took (`durationMs`, in milliseconds).

In [ ]:
progress = query.lastProgress
{key: progress[key] for key in ["batchId", "numInputRows", "inputRowsPerSecond", "processedRowsPerSecond", "durationMs"]}

Every batch is also recorded in `data/metrics/batch_log.csv` by `write_batch()`:

In [ ]:
pd.read_csv(BATCH_LOG)

### Look inside PostgreSQL
Spark can also *read* from PostgreSQL, so the tables can be queried while the stream is still writing to them.

In [ ]:
read_query(spark, """
    SELECT event_time, user_id, event_type, product_name, price, quantity, total_amount, batch_id
    FROM events ORDER BY inserted_at DESC, event_time DESC LIMIT 10
""")

In [ ]:
read_query(spark, """
    SELECT event_type, COUNT(*) AS events, SUM(total_amount) AS total_value
    FROM events GROUP BY event_type ORDER BY events DESC
""")

In [ ]:
read_query(spark, """
    SELECT reject_reason, COUNT(*) AS rows, MIN(event_type) AS example_type,
           MIN(price) AS example_price, MIN(quantity) AS example_quantity
    FROM rejected_events GROUP BY reject_reason ORDER BY rows DESC
""")

A first look at **latency**, the time from an event happening (`event_time`) to its arrival in PostgreSQL (`inserted_at`). Performance is measured properly in Part 4.

In [ ]:
read_query(spark, """
    SELECT ROUND(AVG(EXTRACT(EPOCH FROM inserted_at - event_time))::numeric, 2)::float AS avg_latency_seconds,
           ROUND(MAX(EXTRACT(EPOCH FROM inserted_at - event_time))::numeric, 2)::float AS max_latency_seconds
    FROM events
""")

### Check the results against the answer key
**Stop the generator now** (press Ctrl+C in its terminal). The next cell waits until the stream has processed every row the generator wrote, then compares the tables with the generator's log.

In [ ]:
monitor.wait_until_caught_up()

In [ ]:
monitor.check_results()

### Stop the stream
A streaming query runs until it is stopped. Stopping it here also releases the checkpoint, so `spark_streaming_to_postgres.py` can be used afterwards.

In [ ]:
query.stop()
print("Stream stopped. Still active:", query.isActive)

In [ ]:
spark.stop()